#### E-Commerce Review Intelligence System

**MBA Business Analytics Assignment**  
**Dataset:** Amazon Fine Food Reviews

This notebook develops an end-to-end review intelligence workflow that translates unstructured customer feedback into actionable product and customer-experience insights.

#### Business Problem

E-commerce leaders receive thousands of product reviews, but manually reading them does not scale. The business needs a reliable way to identify how customers feel, understand the themes behind that feedback, and prioritize products where negative experiences create the greatest commercial risk.

**Decision question:** Which products and customer-experience themes should management investigate first to protect customer satisfaction and future revenue?

#### Dataset Description

This analysis uses the Amazon Fine Food Reviews dataset. Each record contains a product identifier, a 1-5 star rating, a review summary, and the full review text. Ratings provide a practical sentiment proxy: scores 1-2 are negative, 3 is neutral, and 4-5 are positive.

The dataset is customer-generated and may contain duplicate, missing, or very short reviews. It is therefore validated and cleaned before modeling. Ratings are a useful proxy, but they may not fully match the sentiment expressed in the written review.

#### Proposed Solution

The proposed review intelligence system combines descriptive analytics and natural language processing. It will:

1. validate and prepare reviews for analysis;
2. translate ratings into three sentiment classes;
3. explore sentiment, rating, and review-length patterns;
4. train a TF-IDF and Logistic Regression baseline for explainable sentiment prediction;
5. discover recurring review themes using Latent Dirichlet Allocation (LDA); and
6. produce a product risk watchlist that managers can use to prioritize action.

#### Why This Method Fits an MBA Business Analytics Assignment

The workflow connects a clear business decision to repeatable analytical evidence. TF-IDF + Logistic Regression is deliberately used as the primary model because it is fast, interpretable, and provides a defensible baseline before considering more complex BiLSTM or Transformer models. LDA complements sentiment analysis: sentiment explains **how** customers feel, while topics explain **what** they are discussing.

#### Project Workflow and Deliverables

This notebook is organised as a reusable review-intelligence workflow:

1. load, validate, and clean Amazon Fine Food Reviews;
2. derive negative, neutral, and positive sentiment from ratings;
3. conduct exploratory analysis of review patterns;
4. train and evaluate an explainable TF-IDF + Logistic Regression baseline;
5. discover discussion themes with LDA topic modelling; and
6. create business-ready insight tables, charts, and reusable model artifacts.

**Optional extensions.** The accompanying project design supports BiLSTM and DistilBERT experiments. The present MBA submission prioritises the interpretable baseline and LDA because they directly support an evidence-based management decision.

In [2]:
from pathlib import Path
import html, json, random, re, warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [21]:
import seaborn as sns

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [3]:
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
warnings.filterwarnings("ignore")

In [4]:
DATA_PATH = Path("Reviews.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError("Place Reviews.csv in the NLP_Assignment folder.")

In [8]:
df = pd.read_csv(DATA_PATH)
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [6]:
df.shape

(568454, 10)

In [7]:
df.columns

Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text'],
      dtype='str')

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 568454 entries, 0 to 568453
Data columns (total 10 columns):
 #   Column                  Non-Null Count   Dtype
---  ------                  --------------   -----
 0   Id                      568454 non-null  int64
 1   ProductId               568454 non-null  str  
 2   UserId                  568454 non-null  str  
 3   ProfileName             568428 non-null  str  
 4   HelpfulnessNumerator    568454 non-null  int64
 5   HelpfulnessDenominator  568454 non-null  int64
 6   Score                   568454 non-null  int64
 7   Time                    568454 non-null  int64
 8   Summary                 568427 non-null  str  
 9   Text                    568454 non-null  str  
dtypes: int64(5), str(5)
memory usage: 43.4 MB


In [5]:
def sentiment(score):
    if score <= 2:
        return 'negative'
    elif score == 3:
        return 'neutral'
    else:
        return 'positive'

In [10]:
df["Sentiment"] = df["Score"].apply(sentiment)
df[["Score", "Sentiment"]].head(10)

,Score,Sentiment
0,5,positive
1,1,negative
2,4,positive
3,2,negative
4,5,positive
5,4,positive
6,5,positive
7,5,positive
8,5,positive
9,5,positive


In [6]:
def replace_dictionary_terms(text, mapping):
    for short, full in mapping.items():
        text = re.sub(rf"\b{re.escape(short)}\b", full, text, flags=re.IGNORECASE)
    return text

In [7]:
CONTRACTIONS = {"don't": "do not", "can't": "cannot", "won't": "will not",
                "isn't": "is not", "wasn't": "was not", "didn't": "did not"}
SLANG = {"idk": "i do not know", "u": "you", "ur": "your", "btw": "by the way"}

#### Text Normalisation

**Why this step matters.** Classical machine-learning models need consistent text inputs. The process below removes non-review noise while retaining meaningful words and explicit negation, such as `do not`.

In [11]:
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,Sentiment
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...,positive
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...,negative
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...,positive
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...,negative
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...,positive


In [10]:
# Keep the fields used for customer, product, sentiment, and text analysis.
analysis_columns = ["ProductId", "Score", "Summary", "Text",
                    "HelpfulnessNumerator", "HelpfulnessDenominator"]
reviews = df[[column for column in analysis_columns if column in df.columns]].copy()

In [11]:
# Remove incomplete reviews and duplicated product-review pairs.
reviews = reviews.dropna(subset=["ProductId", "Score", "Text"])
reviews["Summary"] = reviews["Summary"].fillna("").astype(str)
reviews["Text"] = reviews["Text"].astype(str)
reviews = reviews.drop_duplicates(subset=["ProductId", "Text"])

In [14]:
reviews["sentiment"] = reviews["Score"].apply(sentiment)
reviews["combined_text"] = (reviews["Summary"] + " " + reviews["Text"]).str.strip()
reviews = reviews.reset_index(drop=True)

reviews.head()

,ProductId,Score,Summary,Text,HelpfulnessNumerator,HelpfulnessDenominator,sentiment,combined_text
0,B0042FX6F0,5,Great price,Dealing with Amazon is always a pleasure ! the...,0,2,positive,Great price Dealing with Amazon is always a pl...
1,B001BCUBFK,5,The dog seems to love it,I have a dog who is a very picky eater. I'd be...,6,6,positive,The dog seems to love it I have a dog who is a...
2,B001ELL6KM,4,Polenta,"Living a rural area, I have a hard time findin...",4,5,positive,"Polenta Living a rural area, I have a hard tim..."
3,B000GPXIEY,1,Beware of this product--one of our dogs died.,You should monitor your dog very carefully if ...,5,13,negative,Beware of this product--one of our dogs died. ...
4,B001E6GFKI,5,P19 rocks! Snake-oil theory debunked!,I have been a loyal fan of Product 19 for at l...,5,7,positive,P19 rocks! Snake-oil theory debunked! I have b...


In [15]:
# Use a stratified sample to keep the rating mix representative and manageable.
SAMPLE_SIZE = 50000

if len(reviews) > SAMPLE_SIZE:
    reviews, _ = train_test_split(reviews, train_size=SAMPLE_SIZE,
                                  random_state=RANDOM_STATE, stratify=reviews["Score"])

#### Data Preparation

**Why this step matters.** Real datasets can have missing fields, duplicate reviews, and invalid rows. The preparation process makes the analysis credible by applying consistent quality rules before any model sees the data. A stratified sample preserves the original rating mix while keeping the work reproducible and manageable.

In [9]:
def normalize_text(text):
    text = html.unescape(str(text)).lower()
    text = replace_dictionary_terms(text, CONTRACTIONS)
    text = replace_dictionary_terms(text, SLANG)
    text = re.sub(r"https?://\S+|www\.\S+|<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", re.sub(r"[^a-z\s']", " ", text)).strip()

In [16]:
reviews["clean_text"] = reviews["combined_text"].apply(normalize_text)
reviews = reviews.loc[reviews["clean_text"].str.len() > 0].copy()
reviews["review_length"] = reviews["clean_text"].str.split().str.len()
reviews = reviews.reset_index(drop=True)
reviews[["sentiment", "review_length", "clean_text"]].head()

,sentiment,review_length,clean_text
0,positive,18,great price dealing with amazon is always a pl...
1,positive,121,the dog seems to love it i have a dog who is a...
2,positive,49,polenta living a rural area i have a hard time...
3,negative,85,beware of this product one of our dogs died yo...
4,positive,176,p rocks snake oil theory debunked i have been ...


#### Exploratory Data Analysis

**Why this step matters.** EDA exposes class imbalance, text length, rating spread, and product-level concentration. These findings guide sampling, metrics, and business questions.

In [17]:
dataset_summary = pd.Series({
    "Reviews analysed": len(reviews),
    "Products represented": reviews["ProductId"].nunique(),
    "Average words per review": round(reviews["review_length"].mean(), 1),
    "Median words per review": round(reviews["review_length"].median(), 1),
})
dataset_summary

Reviews analysed            50000.0
Products represented        20836.0
Average words per review       83.4
Median words per review        60.0
dtype: float64

In [ ]:
sentiment_distribution = reviews["sentiment"].value_counts().reindex(
    ["negative", "neutral", "positive"], fill_value=0
)
sentiment_distribution.to_frame("review_count")

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=reviews, x="sentiment", order=["negative", "neutral", "positive"])
plt.title("Review Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(reviews["review_length"].clip(upper=300), bins=50)
plt.title("Review Length Distribution (Capped at 300 Words)")
plt.xlabel("Words per Review")
plt.ylabel("Number of Reviews")
plt.show()

#### Baseline Sentiment Model

**Why this model matters.** TF-IDF + Logistic Regression is fast, interpretable, and a strong benchmark. It shows whether customer sentiment can be predicted reliably before more complex models are considered.

In [18]:
train_df, test_df = train_test_split(
    reviews, test_size=0.20, random_state=RANDOM_STATE,
    stratify=reviews["sentiment"]
)
print(f"Training reviews: {len(train_df):,} | Test reviews: {len(test_df):,}")

Training reviews: 40,000 | Test reviews: 10,000


In [22]:
baseline_model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=50000, ngram_range=(1, 2),
                              min_df=3, max_df=0.95, sublinear_tf=True,
                              stop_words="english")),
    ("classifier", LogisticRegression(max_iter=1200, class_weight="balanced",
                                      random_state=RANDOM_STATE)),
])

In [ ]:
baseline_model.fit(train_df["clean_text"], train_df["sentiment"])
test_predictions = baseline_model.predict(test_df["clean_text"])
test_probabilities = baseline_model.predict_proba(test_df["clean_text"])
print("Baseline model trained successfully.")

In [20]:
baseline_model.fit(train_df["clean_text"], train_df["sentiment"])
test_predictions = baseline_model.predict(test_df["clean_text"])
print("Baseline model trained successfully.")

NameError: name 'baseline_model' is not defined

In [ ]:
baseline_metrics = pd.DataFrame(classification_report(
    test_df["sentiment"], test_predictions, output_dict=True, zero_division=0
)).transpose().round(3)
baseline_metrics

In [ ]:
labels = ["negative", "neutral", "positive"]
confusion = confusion_matrix(test_df["sentiment"], test_predictions, labels=labels)

sns.heatmap(confusion, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.title("TF-IDF + Logistic Regression: Confusion Matrix")
plt.xlabel("Predicted sentiment")
plt.ylabel("Actual sentiment")
plt.show()